<a href="https://colab.research.google.com/github/sis00211/4dong4dong/blob/main/Copy_of_Monthly_Data_to_Google_Sheets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 0. Code Settings

In [3]:
# 1. Force remove everything to clear the corrupted metadata
!pip uninstall -y meteostat numpy pandas

# 2. Reinstall with specific version constraints for Python 3.12 compatibility
!pip install "numpy<2.0" "pandas<2.2.0" meteostat

# 3. This forces Colab to reset its internal 'module finder'
import sys
if 'meteostat' in sys.modules:
    del sys.modules['meteostat']

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: pandas 2.3.3
Uninstalling pandas-2.3.3:
  Successfully uninstalled pandas-2.3.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of meteostat to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 158.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 174.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.1.4 which is incompatible.
mizani 0.13.5 requires pandas>=2.2.0, but you have pandas 2.1.4 which is incompatible.
opencv-contrib-python 4.12.0.88 requires nump

After 'RESTART SESSION', run the cells below

In [1]:
!pip install --upgrade gspread pandas gspread_dataframe oauth2client
!pip install pandas openpyxl xlrd

  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.4 MB)
  Attempting uninstall: pandas
    Found existing installation: pandas 2.1.4
    Uninstalling pandas-2.1.4:
      Successfully uninstalled pandas-2.1.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.


In [ ]:
from meteostat import Daily, Point, Stations
from datetime import datetime
import pandas as pd
import numpy as np
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from gspread_dataframe import set_with_dataframe

In [ ]:
from google.colab import files
# Upload your service account JSON file. You can find the file from NYPL ENERGY & PROJECTS\05_Utility and Energy Data\EC3 Data\Monthly_EC3_Reports/\smooth-loop-395415-6ac31781bb0e.json
uploaded = files.upload()

Saving smooth-loop-395415-6ac31781bb0e.json to smooth-loop-395415-6ac31781bb0e.json


In [ ]:
CREDENTIALS_FILE = "smooth-loop-395415-6ac31781bb0e.json"

In [ ]:
folder_path = '/content/drive/My Drive/EC3_Data/Monthly_EC3_Reports/EC3REPORT'
SHEET_NAME = "Monthly_Data"
import os

files = os.listdir(folder_path)
print("📁 Files in folder:")
for f in files:
    print(f)

In [ ]:
import os

files = os.listdir(folder_path)
print("📁 Files in folder:")
for f in files:
    print(f)

📁 Files in folder:
Estimated Meter Report.xls
Monthly Agency Data.xls
Core Agency Report.pdf
Energy Deviation larger than 25 Percent.xls
Energy Change Report for all Facilities.xls
Monthly Facility Data by Account.xls


In [ ]:
# Authenticate and connect to Google Sheets
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_name(CREDENTIALS_FILE, scope)
client = gspread.authorize(creds)

# Step 1. Energy Deviation larger than 25 Percent File Update

In [ ]:
import pandas as pd
import numpy as np

file_to_read = os.path.join(folder_path, 'Energy Deviation larger than 25 Percent.xls')
df = pd.read_excel(file_to_read)
WORKSHEET_NAME = "Deviation_Raw Data"
#df.head()

In [ ]:
# Columns you want to clean
percent_columns = ['CY_Usage_Deviation_%', 'CY_DMD_Deviation_%','3Y_AVG_Usage_Deviation_%','3Y_AVG_DMD_Deviation_%']

def clean_percent_column(series):
    def convert(val):
        if pd.isna(val):
            return None
        val = str(val).replace('%', '').strip()
        try:
            return float(val) / 100
        except ValueError:
            return None
    return series.apply(convert)

for col in percent_columns:
    df[col] = clean_percent_column(df[col])

df[percent_columns].head()

,CY_Usage_Deviation_%,CY_DMD_Deviation_%,3Y_AVG_Usage_Deviation_%,3Y_AVG_DMD_Deviation_%
0,3.71,2.50,3.94,1.89
1,0.67,0.63,0.63,0.47
2,0.28,0.08,0.21,0.00
3,0.35,0.14,0.35,0.14
4,0.03,0.29,0.14,0.29


In [ ]:
# Load the worksheet
sheet = client.open(SHEET_NAME)
worksheet = sheet.worksheet(WORKSHEET_NAME)

In [ ]:
# Overwrite the sheet with updated data
worksheet.clear()
set_with_dataframe(worksheet, df)

# Step 2. Energy Change Report for all Facilities file Update

In [ ]:
file_to_read = os.path.join(folder_path, 'Energy Change Report for all Facilities.xls')
df = pd.read_excel(file_to_read)
#df.head()

In [ ]:
WORKSHEET_NAME2 = "Change_Raw Data"
worksheet = sheet.worksheet(WORKSHEET_NAME2)

In [ ]:
# Overwrite the sheet with updated data
worksheet.clear()
set_with_dataframe(worksheet,df)

# Step 3. Monthly Agency Data Update

In [ ]:
file_to_read = os.path.join(folder_path, 'Monthly Agency Data.xls')
df = pd.read_excel(file_to_read)

In [ ]:
headers = df.iloc[3].tolist()
data = df.iloc[4:].copy()

data.columns = headers
data = data.dropna(axis=1, how='all')
data = data.dropna(axis=0, how='all')
data.reset_index(drop=True, inplace=True)

data.rename(columns={data.columns[0]: "Index"}, inplace=True)

#Transpose the dataset
transposed_df = data.set_index("Index").T
transposed_df.reset_index(inplace=True)
transposed_df.rename(columns={"index": "Date"}, inplace=True)

In [ ]:
# Convert columns to numeric
transposed_df["Electricity Usage (kWh)"] = pd.to_numeric(transposed_df["Electricity Usage (kWh)"], errors='coerce')
transposed_df["Gas (Therms)"] = pd.to_numeric(transposed_df["Gas (Therms)"], errors='coerce')
transposed_df["Steam (MLbs)"] = pd.to_numeric(transposed_df["Steam (MLbs)"], errors='coerce')

# Calculate mmBTUs
transposed_df["Electricity (mmBTU)"] = transposed_df["Electricity Usage (kWh)"] * 0.003412
transposed_df["Gas (mmBTU)"] = transposed_df["Gas (Therms)"] * 0.1
transposed_df["Steam (mmBTU)"] = transposed_df["Steam (MLbs)"] * 1.194

In [ ]:
#Date -> Fiscal Year
transposed_df["Date"] = pd.to_datetime(transposed_df["Date"], errors="coerce")
transposed_df["Fiscal Year"] = transposed_df["Date"].apply(
    lambda d: d.year + 1 if d.month >= 7 else d.year)

In [ ]:
WORKSHEET_NAME3 = "Agency_Raw_Data"
worksheet = sheet.worksheet(WORKSHEET_NAME3)
worksheet.clear()
set_with_dataframe(worksheet,transposed_df)

In [ ]:
# Step 4. Add CO2 Emissions Columns Using Carbon Conversion Factors
CARBON_SHEET_NAME = "Carbon Conversion"
carbon_ws = sheet.worksheet(CARBON_SHEET_NAME)
carbon_df = pd.DataFrame(carbon_ws.get_all_records())

# Clean up carbon data
carbon_df.columns = [c.strip() for c in carbon_df.columns]
carbon_df.rename(columns={"Fiscal Year": "FY", "Parameter": "Parameter", "Value": "Factor"}, inplace=True)
carbon_df["Factor"] = pd.to_numeric(carbon_df["Factor"], errors="coerce")
carbon_df["FY"] = pd.to_numeric(carbon_df["FY"], errors="coerce")

# Reload Agency_Raw_Data
agency_ws = sheet.worksheet("Agency_Raw_Data")
agency_df = pd.DataFrame(agency_ws.get_all_records())

# Ensure Fiscal Year is numeric
agency_df["Fiscal Year"] = pd.to_numeric(agency_df["Fiscal Year"], errors="coerce")

# Map conversion factors
def get_factor(fy, param):
    val = carbon_df.loc[(carbon_df["FY"] == fy) & (carbon_df["Parameter"] == param), "Factor"]
    return val.values[0] if len(val) > 0 else np.nan

# Temporary factor columns (not kept in final output)
agency_df["_Elec_CO2_factor"] = agency_df["Fiscal Year"].apply(lambda x: get_factor(x, "Electricity Usage (kWh)"))
agency_df["_Gas_CO2_factor"]  = agency_df["Fiscal Year"].apply(lambda x: get_factor(x, "Gas (Therms)"))
agency_df["_Steam_CO2_factor"] = agency_df["Fiscal Year"].apply(lambda x: get_factor(x, "Steam (MLbs)"))

# Calculate CO2 Emissions (metric tons)
agency_df["Electricity_CO2_MT"] = agency_df["Electricity Usage (kWh)"] * agency_df["_Elec_CO2_factor"]
agency_df["Gas_CO2_MT"] = agency_df["Gas (Therms)"] * agency_df["_Gas_CO2_factor"]
agency_df["Steam_CO2_MT"] = agency_df["Steam (MLbs)"] * agency_df["_Steam_CO2_factor"]

# --- Add Total CO2 (metric tons) ---
agency_df["Total_CO2_MT"] = agency_df[["Electricity_CO2_MT", "Gas_CO2_MT", "Steam_CO2_MT"]].sum(axis=1)

# Drop temporary factor columns
agency_df = agency_df.drop(columns=["_Elec_CO2_factor", "_Gas_CO2_factor", "_Steam_CO2_factor"], errors="ignore")

# Reorder columns
cols = list(agency_df.columns)
reorder_cols = [
    "Date", "Fiscal Year",
    "Electricity Usage (kWh)", "Gas (Therms)", "Steam (MLbs)",
    "Electricity (mmBTU)", "Gas (mmBTU)", "Steam (mmBTU)",
    "Electricity_CO2_MT", "Gas_CO2_MT", "Steam_CO2_MT", "Total_CO2_MT"
]
final_cols = [c for c in reorder_cols if c in agency_df.columns] + [c for c in cols if c not in reorder_cols]
agency_df = agency_df[final_cols]

# Upload back to Google Sheets
worksheet = sheet.worksheet("Agency_Raw_Data")
worksheet.clear()
set_with_dataframe(worksheet, agency_df)

Step 4. Temperature Addition

In [ ]:
# ==========================================
# Step 5. Add Monthly Average Outdoor Dry-Bulb Temperature
# ==========================================
!pip install -q meteostat

from meteostat import Daily, Stations
import pandas as pd
import numpy as np

# ----- Helper: fetch nearest station by lat/lon -----
def nearest_station_id(lat, lon):
    s = Stations().nearby(lat, lon).fetch(1)
    if s.empty:
        return None, None
    return s.index[0], s.iloc[0]["name"]

# Candidate NYC locations (Central Park, LaGuardia, JFK)
candidates = [
    {"name_hint": "Central Park", "lat": 40.7812, "lon": -73.9665},
    {"name_hint": "LaGuardia",    "lat": 40.7772, "lon": -73.8726},
    {"name_hint": "JFK",          "lat": 40.6413, "lon": -73.7781},
]

# ----- Load Agency_Raw_Data & determine range -----
agency_ws = sheet.worksheet("Agency_Raw_Data")
agency_df = pd.DataFrame(agency_ws.get_all_records())
agency_df["Date"] = pd.to_datetime(agency_df["Date"], errors="coerce")
agency_df["YearMonth"] = agency_df["Date"].dt.to_period("M").astype(str)

start_date = agency_df["Date"].min()
end_date = agency_df["Date"].max()
if pd.isna(start_date) or pd.isna(end_date):
    raise ValueError("Invalid Date values in Agency_Raw_Data.")

# ----- Helper: fetch daily temps & compute tavg_F -----
def fetch_daily_tavgF(station_id, start_date, end_date):
    if station_id is None:
        return pd.DataFrame()
    df = Daily(station_id, start=start_date, end=end_date).fetch()
    if df.empty:
        return df
    # Fill average temp if missing using (tmin + tmax)/2 when possible
    if "tavg" not in df.columns:
        df["tavg"] = (df.get("tmin") + df.get("tmax")) / 2
    else:
        df["tavg"] = df["tavg"].fillna((df.get("tmin") + df.get("tmax")) / 2)
    # Convert to °F
    df["tavg_F"] = df["tavg"] * 9/5 + 32
    return df[["tavg_F"]].copy()

# ----- Gather unique station IDs -----
unique_stations = {}
for cand in candidates:
    sid, sname = nearest_station_id(cand["lat"], cand["lon"])
    if sid and sid not in unique_stations:
        unique_stations[sid] = sname  # keep first occurrence only

if not unique_stations:
    raise RuntimeError("No suitable weather station found near NYC.")

# ----- Fetch daily series for each unique station -----
daily_series_list = []
coverage_info = []

for sid, sname in unique_stations.items():
    d = fetch_daily_tavgF(sid, start_date, end_date)
    if d.empty:
        continue
    nonnull_days = int(d["tavg_F"].notna().sum())
    coverage_info.append({"station_id": sid, "station_name": sname, "days": nonnull_days})
    # Give each station its own column name
    d = d.rename(columns={"tavg_F": f"tavg_F__{sid}"})
    daily_series_list.append(d)

if not coverage_info:
    raise RuntimeError("No temperature data returned from any NYC stations in the requested range.")

# ----- Log best single station -----
best = max(coverage_info, key=lambda x: x["days"])
print(f"✅ Best single-station coverage: {best['station_name']} ({best['station_id']}), {best['days']} days")

# ----- Blend multiple stations to fill gaps (row-wise mean ignoring NaN) -----
if len(daily_series_list) == 1:
    combined_daily = daily_series_list[0].copy()
    # Rename back to a common name for downstream steps
    only_col = combined_daily.columns[0]
    combined_daily = combined_daily.rename(columns={only_col: "tavg_F"})
else:
    # Outer-join on the date index while avoiding duplicate column names
    combined_daily = daily_series_list[0].copy()
    for d in daily_series_list[1:]:
        # If any overlapping column names are present (shouldn't happen now), skip them
        overlap = set(combined_daily.columns).intersection(set(d.columns))
        if overlap:
            # rename with a random suffix to avoid collision (very unlikely)
            d = d.rename(columns={c: f"{c}_dup" for c in overlap})
        combined_daily = combined_daily.join(d, how="outer")
    # Row-wise mean across all station columns
    combined_daily["tavg_F"] = combined_daily.mean(axis=1, skipna=True)

# ----- Aggregate to monthly average temperature -----
combined_daily["YearMonth"] = combined_daily.index.to_period("M").astype(str)
monthly_temp = (
    combined_daily
    .groupby("YearMonth")[["tavg_F"]]
    .mean()
    .reset_index()
    .rename(columns={"tavg_F": "Avg_DryBulbTemp_F"})
)

# ----- Merge into Agency_Raw_Data -----
merged = agency_df.merge(monthly_temp, on="YearMonth", how="left")

# ----- Reorder columns for final output -----
ordered = [
    "Date", "Fiscal Year",
    "Electricity Usage (kWh)", "Gas (Therms)", "Steam (MLbs)",
    "Electricity (mmBTU)", "Gas (mmBTU)", "Steam (mmBTU)",
    "Electricity_CO2_MT", "Gas_CO2_MT", "Steam_CO2_MT", "Total_CO2_MT",
    "Avg_DryBulbTemp_F"
]
final_cols = [c for c in ordered if c in merged.columns] + [c for c in merged.columns if c not in ordered]
merged = merged[final_cols]

# ----- Write back to Google Sheet -----
worksheet = sheet.worksheet("Agency_Raw_Data")
worksheet.clear()
set_with_dataframe(worksheet, merged)

print("✅ Step 5 Complete — Monthly Avg_DryBulbTemp_F added (auto-selected & blended NYC stations; duplicates de-duped).")